# Task별 Dataset 추출기

raw_v1 merged dataset에서 특정 task만 필터링하여 새로운 Dataset을 생성한다.

In [16]:
from pathlib import Path
from datasets import load_from_disk

# ── 설정 ──
TASK_NAME = "smol-molecule_captioning"
REPR = "SELFIES"  # "SMILES" or "SELFIES"

PROJECT_ROOT = Path("/opt/11-MolDA/New_MolDA")
SRC_ROOT = PROJECT_ROOT / "dataset" / "Raw" / REPR / "raw_v1"
DST_ROOT = PROJECT_ROOT / "dataset" / "Processed" / REPR / TASK_NAME

print(f"Source: {SRC_ROOT}")
print(f"Dest:   {DST_ROOT}")

Source: /opt/11-MolDA/New_MolDA/dataset/Raw/SELFIES/raw_v1
Dest:   /opt/11-MolDA/New_MolDA/dataset/Processed/SELFIES/smol-molecule_captioning


In [17]:
# ── Raw 합본 (Step 3 완료)에서 task 필터링 후 저장 ──
import time
import numpy as np

for split_name in ["Train", "Val", "Test"]:
    src_path = SRC_ROOT / split_name
    if not src_path.exists():
        print(f"[SKIP] {src_path} not found")
        continue

    t0 = time.time()
    ds = load_from_disk(str(src_path))
    print(f"[{split_name}] load_from_disk: {time.time()-t0:.1f}s ({len(ds):,} rows)")

    t1 = time.time()
    task_array = np.array(ds["task"])
    indices = np.where(task_array == TASK_NAME)[0].tolist()
    print(f"[{split_name}] numpy filter:   {time.time()-t1:.1f}s ({len(indices):,} matched)")

    t2 = time.time()
    filtered = ds.select(indices)
    print(f"[{split_name}] ds.select:      {time.time()-t2:.1f}s")

    dst_path = DST_ROOT / split_name
    dst_path.mkdir(parents=True, exist_ok=True)

    t3 = time.time()
    filtered.save_to_disk(str(dst_path))
    print(f"[{split_name}] save_to_disk:   {time.time()-t3:.1f}s")

    print(f"[{split_name}] DONE — {len(filtered):>6,} rows saved (total {time.time()-t0:.1f}s)\n")

[Train] load_from_disk: 0.2s (3,219,141 rows)
[Train] numpy filter:   1.7s (50,595 matched)
[Train] ds.select:      0.1s


Saving the dataset (1/1 shards): 100%|██████████| 50595/50595 [00:00<00:00, 95466.95 examples/s]


[Train] save_to_disk:   0.5s
[Train] DONE — 50,595 rows saved (total 2.4s)

[Val] load_from_disk: 0.1s (36,016 rows)
[Val] numpy filter:   0.1s (1,269 matched)
[Val] ds.select:      0.0s


Saving the dataset (1/1 shards): 100%|██████████| 1269/1269 [00:00<00:00, 111879.85 examples/s]


[Val] save_to_disk:   0.0s
[Val] DONE —  1,269 rows saved (total 0.1s)

[Test] load_from_disk: 0.0s (32,821 rows)
[Test] numpy filter:   0.0s (2,538 matched)
[Test] ds.select:      0.0s


Saving the dataset (1/1 shards): 100%|██████████| 2538/2538 [00:00<00:00, 124189.41 examples/s]

[Test] save_to_disk:   0.0s
[Test] DONE —  2,538 rows saved (total 0.0s)



In [18]:
# ── 검증: 저장된 dataset 로드 및 prompt_text 존재 확인 ──
for split in ["Train", "Val", "Test"]:
    ds = load_from_disk(str(DST_ROOT / split))
    print(f"{split}: {len(ds):>6,} rows | columns: {ds.column_names}")
    assert "prompt_text" in ds.column_names, f"prompt_text missing in {split}!"

print(f"\n--- Sample (Train[0]) ---")
ds_train = load_from_disk(str(DST_ROOT / "Train"))
for k in ["task", "prompt_text", "target_text"]:
    print(f"  {k}: {repr(ds_train[0][k])[:200]}")

Train: 50,595 rows | columns: ['x', 'edge_index', 'edge_attr', 'label', 'input_mol_string', 'task_subtask_pair', 'instruction', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'task', 'prompt_text', 'target_text']
Val:  1,269 rows | columns: ['x', 'edge_index', 'edge_attr', 'label', 'input_mol_string', 'task_subtask_pair', 'instruction', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'task', 'prompt_text', 'target_text']
Test:  2,538 rows | columns: ['x', 'edge_index', 'edge_attr', 'label', 'input_mol_string', 'task_subtask_pair', 'instruction', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'task', 'prompt_text', 'target_text']

--- Sample (Train[0]) ---
  task: 'smol-molecule_captioning'
  prompt_text: '<|startoftext|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regr
  target_text: '<DESCRIPTION>The

## Toy30 Dataset 생성 (전체 task, task당 30개)

raw_v1 merged 데이터셋에서 **모든 task별로 30개씩** 샘플링하여 toy30 버전을 생성한다.
- Source: `dataset/Raw/SELFIES/raw_v1/{Train,Val,Test}` (전체 task merged)
- Dest: `dataset/Processed/SELFIES/toy30/{Train,Val,Test}`
- 각 split 내 모든 unique task에서 균등하게 30개씩 추출 (task 샘플 수 < 30이면 전부 사용)

In [14]:
from pathlib import Path
from datasets import load_from_disk, concatenate_datasets
import numpy as np
import time

# ── 설정 ──
TOY_SIZE = 7    # task당 최대 샘플 수
SEED = 42
REPR = "SELFIES"

PROJECT_ROOT = Path("/opt/11-MolDA/New_MolDA")
SRC_ROOT = PROJECT_ROOT / "dataset" / "Raw" / REPR / "raw_v1"      # merged raw dataset
DST_ROOT = PROJECT_ROOT / "dataset" / "Processed" / REPR / "toy7" # toy7 출력 경로

for split_name in ["Train", "Val", "Test"]:
    src_path = SRC_ROOT / split_name
    if not src_path.exists():
        print(f"[SKIP] {src_path} not found")
        continue

    t0 = time.time()
    ds = load_from_disk(str(src_path))
    print(f"[{split_name}] loaded {len(ds):,} rows ({time.time()-t0:.1f}s)")

    # task 컬럼을 numpy로 변환하여 빠르게 필터링
    task_array = np.array(ds["task"])
    unique_tasks = sorted(set(task_array))
    print(f"[{split_name}] {len(unique_tasks)} unique tasks found")

    # 각 task별로 TOY_SIZE개씩 샘플링
    rng = np.random.default_rng(SEED)
    sampled_indices = []
    for task_name in unique_tasks:
        task_indices = np.where(task_array == task_name)[0]
        n = min(TOY_SIZE, len(task_indices))
        chosen = rng.choice(task_indices, size=n, replace=False)
        sampled_indices.extend(chosen.tolist())
        print(f"  {task_name}: {len(task_indices):>8,} → {n}")

    # 셔플 후 선택
    rng.shuffle(sampled_indices)
    ds_toy = ds.select(sampled_indices)

    # 저장
    dst_path = DST_ROOT / split_name
    dst_path.mkdir(parents=True, exist_ok=True)
    ds_toy.save_to_disk(str(dst_path))
    print(f"[{split_name}] DONE — {len(ds_toy):,} rows saved ({time.time()-t0:.1f}s)\n")

[Train] loaded 3,219,141 rows (0.1s)
[Train] 21 unique tasks found
  bace:    1,210 → 7
  chebi-20-mol2text:    9,568 → 7
  chebi-20-text2mol:   26,402 → 7
  forward_reaction_prediction:   28,106 → 7
  qm9_homo:  117,656 → 7
  qm9_homo_lumo_gap:  117,538 → 7
  qm9_lumo:  117,706 → 7
  reagent_prediction:  121,896 → 7
  retrosynthesis:    2,715 → 7
  smol-forward_synthesis:  969,054 → 7
  smol-molecule_captioning:   50,595 → 7
  smol-molecule_generation:   56,498 → 7
  smol-name_conversion-i2s:  299,890 → 7
  smol-name_conversion-s2i:  299,890 → 7
  smol-property_prediction-bbbp:    1,569 → 7
  smol-property_prediction-clintox:    1,144 → 7
  smol-property_prediction-esol:      888 → 7
  smol-property_prediction-hiv:   32,863 → 7
  smol-property_prediction-lipo:    3,360 → 7
  smol-property_prediction-sider:   22,820 → 7
  smol-retrosynthesis:  937,773 → 7


Saving the dataset (1/1 shards): 100%|██████████| 147/147 [00:00<00:00, 1174.99 examples/s]


[Train] DONE — 147 rows saved (2.9s)

[Val] loaded 36,016 rows (0.0s)
[Val] 19 unique tasks found
  bace:      151 → 7
  chebi-20-mol2text:    3,299 → 7
  chebi-20-text2mol:    3,299 → 7
  forward_reaction_prediction:    2,488 → 7
  qm9_homo:    2,402 → 7
  qm9_homo_lumo_gap:    2,399 → 7
  qm9_lumo:    2,403 → 7
  reagent_prediction:    2,488 → 7
  retrosynthesis:    2,574 → 7
  smol-forward_synthesis:    2,049 → 7
  smol-molecule_captioning:    1,269 → 7
  smol-molecule_generation:    1,269 → 7
  smol-property_prediction-bbbp:      196 → 7
  smol-property_prediction-clintox:      143 → 7
  smol-property_prediction-esol:      111 → 7
  smol-property_prediction-hiv:    4,104 → 7
  smol-property_prediction-lipo:      420 → 7
  smol-property_prediction-sider:    2,860 → 7
  smol-retrosynthesis:    2,092 → 7


Saving the dataset (1/1 shards): 100%|██████████| 133/133 [00:00<00:00, 12909.13 examples/s]


[Val] DONE — 133 rows saved (0.2s)

[Test] loaded 32,821 rows (0.0s)
[Test] 19 unique tasks found
  bace:      152 → 7
  chebi-20-mol2text:    3,297 → 7
  chebi-20-text2mol:    3,297 → 7
  forward_reaction_prediction:    1,000 → 7
  qm9_homo:      684 → 7
  qm9_homo_lumo_gap:      661 → 7
  qm9_lumo:      642 → 7
  reagent_prediction:    1,000 → 7
  retrosynthesis:    1,000 → 7
  smol-forward_synthesis:    4,062 → 7
  smol-molecule_captioning:    2,538 → 7
  smol-molecule_generation:    2,493 → 7
  smol-property_prediction-bbbp:      197 → 7
  smol-property_prediction-clintox:      144 → 7
  smol-property_prediction-esol:      112 → 7
  smol-property_prediction-hiv:    4,106 → 7
  smol-property_prediction-lipo:      420 → 7
  smol-property_prediction-sider:    2,860 → 7
  smol-retrosynthesis:    4,156 → 7


Saving the dataset (1/1 shards): 100%|██████████| 133/133 [00:00<00:00, 16713.88 examples/s]

[Test] DONE — 133 rows saved (0.0s)



In [15]:
# ── 검증: task별 샘플 수 확인 ──
from collections import Counter

for split in ["Train", "Val", "Test"]:
    ds = load_from_disk(str(DST_ROOT / split))
    task_counts = Counter(ds["task"])
    print(f"\n{split}: {len(ds):,} rows total | {len(task_counts)} tasks")
    for task, cnt in sorted(task_counts.items()):
        print(f"  {task}: {cnt}")

# 샘플 확인
print(f"\n--- Sample (Train[0]) ---")
ds_train = load_from_disk(str(DST_ROOT / "Train"))
for k in ["task", "prompt_text", "target_text"]:
    print(f"  {k}: {repr(ds_train[0][k])[:200]}")


Train: 147 rows total | 21 tasks
  bace: 7
  chebi-20-mol2text: 7
  chebi-20-text2mol: 7
  forward_reaction_prediction: 7
  qm9_homo: 7
  qm9_homo_lumo_gap: 7
  qm9_lumo: 7
  reagent_prediction: 7
  retrosynthesis: 7
  smol-forward_synthesis: 7
  smol-molecule_captioning: 7
  smol-molecule_generation: 7
  smol-name_conversion-i2s: 7
  smol-name_conversion-s2i: 7
  smol-property_prediction-bbbp: 7
  smol-property_prediction-clintox: 7
  smol-property_prediction-esol: 7
  smol-property_prediction-hiv: 7
  smol-property_prediction-lipo: 7
  smol-property_prediction-sider: 7
  smol-retrosynthesis: 7

Val: 133 rows total | 19 tasks
  bace: 7
  chebi-20-mol2text: 7
  chebi-20-text2mol: 7
  forward_reaction_prediction: 7
  qm9_homo: 7
  qm9_homo_lumo_gap: 7
  qm9_lumo: 7
  reagent_prediction: 7
  retrosynthesis: 7
  smol-forward_synthesis: 7
  smol-molecule_captioning: 7
  smol-molecule_generation: 7
  smol-property_prediction-bbbp: 7
  smol-property_prediction-clintox: 7
  smol-property_pre